# Book Genre Classification — Training Notebook

This notebook fine-tunes DistilBERT on the Goodreads book genre dataset.
Designed to run on **Kaggle** with GPU T4 x2 accelerator.

**Two training runs** with different hyperparameters are logged to W&B.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets wandb scikit-learn huggingface-hub

## 2. Authenticate W&B and Hugging Face

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

secrets = UserSecretsClient()
WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

wandb.login(key=WANDB_API_KEY)
login(token=HF_TOKEN)

## 3. Load and Prepare Dataset

In [ ]:
import json
import string

import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Load dataset
dataset = load_dataset("pszemraj/goodreads-bookgenres", split="train")
df = pd.DataFrame(dataset)

# Get genre columns
genre_cols = [c for c in df.columns if c not in ("title", "desc")]

# Convert multi-label to single-label
def get_first_genre(row):
    for col in genre_cols:
        if row[col] == 1:
            return col
    return None

df["genre"] = df.apply(get_first_genre, axis=1)
df = df.dropna(subset=["genre", "desc"])

# Clean text
df["desc"] = df["desc"].str.lower()
translator = str.maketrans("", "", string.punctuation)
df["desc"] = df["desc"].str.translate(translator)
df = df.drop_duplicates(subset=["desc"])

# Encode labels
sorted_labels = sorted(df["genre"].unique())
label2id = {label: idx for idx, label in enumerate(sorted_labels)}
id2label = {str(idx): label for idx, label in enumerate(sorted_labels)}
df["label"] = df["genre"].map(label2id)

# Save id2label
with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

# Stratified split
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Number of classes: {len(id2label)}")

## 4. Tokenize Data

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Convert to HF datasets
train_dataset = Dataset.from_pandas(train_df[["desc", "label"]].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[["desc", "label"]].reset_index(drop=True))

def tokenize_fn(examples):
    return tokenizer(examples["desc"], truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## 5. Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

## 6. Training Run 1 — Conservative

- Learning rate: 2e-5
- Epochs: 3
- Batch size: 16
- Warmup steps: 100
- Weight decay: 0.01

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load fresh model for Run 1
model_run1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

training_args_run1 = TrainingArguments(
    output_dir="./results-run1",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    warmup_steps=100,
    weight_decay=0.01,
    report_to="wandb",
    run_name="run-v1-conservative",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_run1 = Trainer(
    model=model_run1,
    args=training_args_run1,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer_run1.train()

## 7. Push Run 1 Model to HF Hub

In [ ]:
HF_REPO = "VikasVishwakarma22/distilbert-goodreads-pipeline"

trainer_run1.model.push_to_hub(HF_REPO, commit_message="Run 1: conservative hyperparams")
tokenizer.push_to_hub(HF_REPO)

wandb.run.summary["hf_model_url"] = f"https://huggingface.co/{HF_REPO}"
wandb.finish()

## 8. Training Run 2 — Aggressive

- Learning rate: 5e-5
- Epochs: 5
- Batch size: 32
- Warmup steps: 200
- Weight decay: 0.05

In [ ]:
# Load fresh model for Run 2
model_run2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

training_args_run2 = TrainingArguments(
    output_dir="./results-run2",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    warmup_steps=200,
    weight_decay=0.05,
    report_to="wandb",
    run_name="run-v2-aggressive",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_run2 = Trainer(
    model=model_run2,
    args=training_args_run2,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer_run2.train()

## 9. Push Run 2 Model to HF Hub (if better)

In [ ]:
# Push Run 2 model (overwrites if better)
trainer_run2.model.push_to_hub(HF_REPO, commit_message="Run 2: aggressive hyperparams")
tokenizer.push_to_hub(HF_REPO)

wandb.run.summary["hf_model_url"] = f"https://huggingface.co/{HF_REPO}"
wandb.finish()

print(f"\nModels pushed to: https://huggingface.co/{HF_REPO}")
print("Check W&B dashboard for run comparisons.")